# Búsqueda de hiperparámetros — Clasificador de Somnolencia (MLP)
**Programacion Paralela y Computacion Distribuida · Universidad de Pamplona · 2026-I**

Notebook dedicado a la **búsqueda en grilla** de hiperparámetros del MLP de una capa
oculta. Usa TensorFlow en CPU (misma arquitectura y modo full-batch que
`etapa2_cuda/cpu_baseline/base-model-cpu.ipynb`) para explorar el espacio de forma
rápida antes de entrenar con los kernels CUDA.

**Espacio de búsqueda (guía del proyecto):**

| Hiperparámetro | Rango | Valores probados |
|----------------|-------|------------------|
| Épocas | 20–50 | 20, 40, 45, 50 |
| Tasa de aprendizaje | 0.01–0.1 | 0.01, 0.08, 0.1 |
| Neuronas ocultas | 64–128 | 64, 75, 78, 128 |

Se elige la combinación con mayor **accuracy en validación** (`dataset/procesado/val.csv`).
El conjunto de test queda reservado para la evaluación final (Phase 5).

**Contenido:**
1. Configuración (entorno, rutas, semilla)
2. Carga del dataset
3. Arquitectura del modelo
4. Búsqueda en grilla (48 combinaciones)
5. Visualización de resultados
6. Exportación (`hyperparameter_results.csv`, `hyperparameter_report.md`)

---
## 1. Configuración

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Albonire/clasificador-imagenes-openmp-cuda.git"
REPO_DIR = "/content/clasificador-imagenes-openmp-cuda"

if IN_COLAB and not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

if IN_COLAB:
    os.chdir(os.path.join(REPO_DIR, "notebooks"))

print(f"En Colab: {IN_COLAB}")
print(f"Directorio de trabajo: {os.getcwd()}")

In [ ]:
!pip install -q tensorflow matplotlib scikit-learn

In [ ]:
import datetime
import itertools
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# CPU-only: la grilla corre en TensorFlow para iterar rápido sin depender de GPU.
tf.config.set_visible_devices([], "GPU")

print(f"TensorFlow version: {tf.__version__}")
print("Dispositivos visibles:")
for device in tf.config.get_visible_devices():
    print(f"  {device}")

In [ ]:
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(REPO_ROOT, "dataset")):
    REPO_ROOT = os.getcwd()

PATHS = {
    "procesado": os.path.join(REPO_ROOT, "dataset", "procesado"),
    "evidencias": os.path.join(REPO_ROOT, "reporte", "evidencias"),
    "notebooks": os.path.join(REPO_ROOT, "notebooks"),
}

for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"  {'OK' if os.path.isdir(path) else 'WARN'} {name}: {path}")

TRAIN_CSV = os.path.join(PATHS["procesado"], "train.csv")
VAL_CSV = os.path.join(PATHS["procesado"], "val.csv")
RESULTS_CSV = os.path.join(PATHS["notebooks"], "hyperparameter_results.csv")
REPORT_MD = os.path.join(PATHS["notebooks"], "hyperparameter_report.md")

print(f"\nTrain CSV: {TRAIN_CSV}")
print(f"Val CSV:   {VAL_CSV}")

---
## 2. Carga del dataset

Particiones generadas en `dataset_preparation.ipynb` (70/15/15, estratificadas).
Formato: `label, pixel_0 .. pixel_4095`.

In [ ]:
TARGET = "label"


def load_split(path):
    df = pd.read_csv(path)
    feature_cols = [c for c in df.columns if c != TARGET]
    x = df[feature_cols].to_numpy(dtype=np.float32)
    y = df[TARGET].to_numpy(dtype=np.float32)
    return x, y


X_train, y_train = load_split(TRAIN_CSV)
X_val, y_val = load_split(VAL_CSV)

N_FEATURES = X_train.shape[1]
assert N_FEATURES == 4096, f"Se esperaban 4096 features (64x64), se obtuvieron {N_FEATURES}"

for name, x, y in [("Train", X_train, y_train), ("Val", X_val, y_val)]:
    print(f"{name}: {x.shape}, positivos(clase1)={int(y.sum())} ({y.mean() * 100:.1f}%)")

print(f"\nFeatures: {N_FEATURES}")
print(f"Rango de pixeles: [{X_train.min():.4f}, {X_train.max():.4f}]")

---
## 3. Arquitectura del modelo

```
Entrada(4096) -> Densa(oculta, ReLU) -> Densa(1, Sigmoide)
```

- **Pérdida:** Binary Cross-Entropy
- **Optimizador:** SGD
- **Modo:** full-batch (todo el train set por época, comparable con `train_gpu.cu`)

In [ ]:
def build_model(hidden_units, learning_rate):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(N_FEATURES,)),
        tf.keras.layers.Dense(hidden_units, activation="relu", name="hidden"),
        tf.keras.layers.Dense(1, activation="sigmoid", name="output"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


build_model(64, 0.1).summary()

---
## 4. Búsqueda de hiperparámetros

Grilla cartesiana dentro de los rangos del proyecto. Cada combinación se entrena
desde cero con la misma semilla; se registra accuracy y pérdida en validación.

In [ ]:
# Rangos: épocas 20-50, lr 0.01-0.1, neuronas 64-128
HIDDEN_GRID = [64, 75, 78, 128]
LR_GRID = [0.01, 0.08, 0.1]
EPOCHS_GRID = [20, 40, 45, 50]

n_combos = len(HIDDEN_GRID) * len(LR_GRID) * len(EPOCHS_GRID)
print(f"Combinaciones a evaluar: {n_combos}")
print(f"  hidden_units: {HIDDEN_GRID}")
print(f"  learning_rate: {LR_GRID}")
print(f"  epochs: {EPOCHS_GRID}")

In [ ]:
results = []

for hidden_units, learning_rate, epochs in itertools.product(HIDDEN_GRID, LR_GRID, EPOCHS_GRID):
    tf.random.set_seed(SEED)
    model = build_model(hidden_units, learning_rate)

    start = time.perf_counter()
    model.fit(
        X_train, y_train,
        batch_size=X_train.shape[0],
        epochs=epochs,
        validation_data=(X_val, y_val),
        verbose=0,
    )
    elapsed = time.perf_counter() - start

    val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
    results.append({
        "hidden_units": hidden_units,
        "learning_rate": learning_rate,
        "epochs": epochs,
        "train_time_sec": elapsed,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
    })
    print(
        f"hidden={hidden_units:3d} lr={learning_rate:<5} epochs={epochs:3d} "
        f"-> val_acc={val_accuracy:.4f} val_loss={val_loss:.4f} time={elapsed:.2f}s"
    )

results_df = pd.DataFrame(results).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
results_df

In [ ]:
best = results_df.iloc[0]
BEST_HIDDEN = int(best["hidden_units"])
BEST_LR = float(best["learning_rate"])
BEST_EPOCHS = int(best["epochs"])

print("Mejor combinación (mayor accuracy en validación):")
print(f"  hidden_units  = {BEST_HIDDEN}")
print(f"  learning_rate = {BEST_LR}")
print(f"  epochs        = {BEST_EPOCHS}")
print(f"  val_accuracy  = {best['val_accuracy']:.4f}")
print(f"  val_loss      = {best['val_loss']:.4f}")

---
## 5. Visualización

Mapas de calor de `val_accuracy` para cada número de épocas (hidden × learning rate).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for ax, epochs in zip(axes, EPOCHS_GRID):
    subset = results_df[results_df["epochs"] == epochs]
    pivot = subset.pivot(index="hidden_units", columns="learning_rate", values="val_accuracy")
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis", vmin=results_df["val_accuracy"].min(), vmax=results_df["val_accuracy"].max())
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{lr:g}" for lr in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("learning_rate")
    ax.set_ylabel("hidden_units")
    ax.set_title(f"val_accuracy — epochs={epochs}")
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, f"{pivot.values[i, j]:.3f}", ha="center", va="center", color="white", fontsize=9)

fig.colorbar(im, ax=axes, shrink=0.6, label="val_accuracy")
fig.suptitle("Búsqueda de hiperparámetros — accuracy en validación", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
top10 = results_df.head(10)
fig, ax = plt.subplots(figsize=(10, 5))
labels = [
    f"h={int(r.hidden_units)} lr={r.learning_rate:g} e={int(r.epochs)}"
    for r in top10.itertuples()
]
ax.barh(labels[::-1], top10["val_accuracy"].values[::-1], color="steelblue")
ax.set_xlabel("val_accuracy")
ax.set_title("Top 10 combinaciones")
ax.set_xlim(results_df["val_accuracy"].min() - 0.02, results_df["val_accuracy"].max() + 0.01)
plt.tight_layout()
plt.show()

---
## 6. Exportación

Los resultados se guardan en `notebooks/` para usarlos en los notebooks de
entrenamiento CPU/GPU y en el reporte CRISP-DM.

In [ ]:
results_df.to_csv(RESULTS_CSV, index=False)
print(f"CSV guardado: {RESULTS_CSV}")

timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
with open(REPORT_MD, "w", encoding="utf-8") as f:
    f.write("# Búsqueda de hiperparámetros (MLP)\n\n")
    f.write(f"Generado: {timestamp}\n\n")
    f.write("## Espacio explorado\n\n")
    f.write("| Hiperparámetro | Rango | Valores |\n")
    f.write("|----------------|-------|---------|\n")
    f.write(f"| Épocas | 20–50 | {', '.join(map(str, EPOCHS_GRID))} |\n")
    f.write(f"| Learning rate | 0.01–0.1 | {', '.join(map(str, LR_GRID))} |\n")
    f.write(f"| Neuronas ocultas | 64–128 | {', '.join(map(str, HIDDEN_GRID))} |\n\n")
    f.write(f"Combinaciones evaluadas: **{n_combos}** (TensorFlow CPU, full-batch)\n\n")
    f.write("## Mejor configuración (validación)\n\n")
    f.write(f"| Métrica | Valor |\n")
    f.write(f"|---------|-------|\n")
    f.write(f"| hidden_units | {BEST_HIDDEN} |\n")
    f.write(f"| learning_rate | {BEST_LR} |\n")
    f.write(f"| epochs | {BEST_EPOCHS} |\n")
    f.write(f"| val_accuracy | {best['val_accuracy']:.4f} |\n")
    f.write(f"| val_loss | {best['val_loss']:.4f} |\n\n")
    f.write("## Tabla completa (ordenada por val_accuracy)\n\n")
    f.write("| hidden_units | learning_rate | epochs | train_time_sec | val_loss | val_accuracy |\n")
    f.write("|-------------:|--------------:|-------:|---------------:|---------:|-------------:|\n")
    for row in results_df.itertuples():
        f.write(
            f"| {int(row.hidden_units)} | {row.learning_rate} | {int(row.epochs)} | "
            f"{row.train_time_sec:.2f} | {row.val_loss:.4f} | {row.val_accuracy:.4f} |\n"
        )

print(f"Reporte guardado: {REPORT_MD}")